<a href="https://colab.research.google.com/github/Ps1012/telco-customer-churn-analysis/blob/main/customer_churn_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn Analysis
---

## 1. Setup

Loading all the libraries we need for this project. Each one has a specific job:
- `pandas` and `numpy` for data manipulation
- `plotly` for interactive charts
- `sklearn` for machine learning models and evaluation
- `xgboost` for the XGBoost classifier
- `imblearn` for handling class imbalance with SMOTE

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, accuracy_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

BLUE   = '#185FA5'
GREEN  = '#1A7A4A'
RED    = '#B83232'
ORANGE = '#E07B00'
GREY   = '#6B7280'

LAYOUT = dict(
    font        = dict(family='Arial, sans-serif', size=12, color='#1A1A2E'),
    paper_bgcolor = 'white',
    plot_bgcolor  = '#F8F9FA',
    title_font  = dict(size=15, color='#1A1A2E'),
    title_x     = 0.05,
    margin      = dict(t=70, b=50, l=60, r=30),
    hoverlabel  = dict(bgcolor='white', font_size=12),
    legend      = dict(bgcolor='white', bordercolor='#E5E7EB', borderwidth=1),
)

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## 2. Load the Data

Loading the CSV and taking a first look at its shape, column types, and a few rows of raw data. This is always the first step — you want to understand what you're working with before doing anything else.

In [10]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('Column types:')
print(df.dtypes.to_string())

Shape   : 7,043 rows x 21 columns
Memory  : 6984.7 KB

Column types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object


In [11]:
# Preview the first few rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
# Summary statistics for all columns
df.describe(include='all')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,3186-AJIEK,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


---
## 3. Data Cleaning

Before any analysis, we need to fix data quality issues. The most important finding here is that `TotalCharges` is stored as a string instead of a number — which hides 11 null values that only appear after we convert it. We also check for duplicate rows and confirm the dataset is otherwise clean.

In [13]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

nulls = df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]
print(f'Null values in TotalCharges after conversion: {df["TotalCharges"].isnull().sum()}')
print()
print('Rows with missing TotalCharges:')
print(nulls.to_string(index=False))
print()
print('All 11 rows have tenure = 0 — brand new customers who have not been billed yet.')
print('Filling with 0 makes business sense here.')

Null values in TotalCharges after conversion: 11

Rows with missing TotalCharges:
customerID  tenure  MonthlyCharges  TotalCharges
4472-LVYGI       0           52.55           NaN
3115-CZMZD       0           20.25           NaN
5709-LVOEQ       0           80.85           NaN
4367-NUYAO       0           25.75           NaN
1371-DWPAZ       0           56.05           NaN
7644-OMVMY       0           19.85           NaN
3213-VVOLG       0           25.35           NaN
2520-SGTTA       0           20.00           NaN
2923-ARZLG       0           19.70           NaN
4075-WKNIU       0           73.35           NaN
2775-SEFEE       0           61.90           NaN

All 11 rows have tenure = 0 — brand new customers who have not been billed yet.
Filling with 0 makes business sense here.


In [15]:
#filling null values
df['TotalCharges'].fillna(0, inplace=True)

# Check for duplicates
print(f'Duplicate rows      : {df.duplicated().sum()}')
print(f'Duplicate customerIDs: {df["customerID"].duplicated().sum()}')
print()
print('Data is clean — no duplicates, all nulls resolved.')

Duplicate rows      : 0
Duplicate customerIDs: 0

Data is clean — no duplicates, all nulls resolved.


---
## 4. Exploratory Data Analysis

Now we explore the data visually to understand what drives churn. Each chart below is answering a specific question. We start with the target variable (churn itself), then look at how numeric and categorical features behave differently for churned vs retained customers.

### 4.1 Churn Distribution

First thing to check: how many customers actually churned? This tells us if we have a class imbalance problem — which affects how we build and evaluate the model.

In [16]:
churn_counts = df['Churn'].value_counts().reset_index()
churn_counts.columns = ['Churn', 'Count']
churn_counts['Percentage'] = (churn_counts['Count'] / len(df) * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    subplot_titles=['Customer Count by Churn Status', 'Churn Proportion']
)

fig.add_trace(go.Bar(
    x=churn_counts['Churn'],
    y=churn_counts['Count'],
    marker_color=[GREEN, RED],
    text=[f'{c:,} ({p}%)' for c, p in zip(churn_counts['Count'], churn_counts['Percentage'])],
    textposition='outside',
    textfont=dict(size=12),
    showlegend=False,
    hovertemplate='<b>Churn: %{x}</b><br>Count: %{y:,}<extra></extra>',
    width=0.4,
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=churn_counts['Churn'],
    values=churn_counts['Count'],
    hole=0.45,
    marker_colors=[GREEN, RED],
    textinfo='label+percent',
    textfont_size=12,
    pull=[0, 0.05],
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>%{percent}<extra></extra>',
), row=1, col=2)

fig.update_layout(**LAYOUT, title_text='Customer Churn Distribution', showlegend=False, height=420)
fig.update_yaxes(title_text='Number of Customers', row=1, col=1, gridcolor='#E5E7EB')
fig.update_xaxes(row=1, col=1, gridcolor='#E5E7EB')
fig.show()

print('Churn = No  :', churn_counts[churn_counts['Churn']=='No']['Count'].values[0], '(73.5%)')
print('Churn = Yes :', churn_counts[churn_counts['Churn']=='Yes']['Count'].values[0], '(26.5%)')
print()
print('We have a class imbalance — 26.5% churn rate.')
print('A model that always predicts No would score 73.5% accuracy but catch zero churners.')
print('We will use F1-Score and ROC-AUC instead of accuracy.')

Churn = No  : 5174 (73.5%)
Churn = Yes : 1869 (26.5%)

We have a class imbalance — 26.5% churn rate.
A model that always predicts No would score 73.5% accuracy but catch zero churners.
We will use F1-Score and ROC-AUC instead of accuracy.


### 4.2 Numeric Features vs Churn

Looking at how `tenure`, `MonthlyCharges`, and `TotalCharges` are distributed differently between churned and retained customers. Overlapping histograms and box plots let us see both the shape and the spread.

In [17]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
colors_churn = {'No': GREEN, 'Yes': RED}

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Tenure — Distribution',         'Monthly Charges — Distribution', 'Total Charges — Distribution',
        'Tenure — Box Plot',             'Monthly Charges — Box Plot',     'Total Charges — Box Plot',
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.09
)

for i, col in enumerate(num_cols, 1):
    for label in ['No', 'Yes']:
        subset = df[df['Churn'] == label][col]

        fig.add_trace(go.Histogram(
            x=subset,
            name=f'Churn = {label}',
            marker_color=colors_churn[label],
            opacity=0.65,
            nbinsx=30,
            showlegend=(i == 1),
            legendgroup=label,
            hovertemplate=f'<b>{col}</b><br>Value: %{{x}}<br>Count: %{{y}}<extra>Churn={label}</extra>',
        ), row=1, col=i)

        fig.add_trace(go.Box(
            y=subset,
            name=f'Churn = {label}',
            marker_color=colors_churn[label],
            boxmean='sd',
            showlegend=False,
            legendgroup=label,
            hovertemplate=f'<b>{col}</b><br>%{{y}}<extra>Churn={label}</extra>',
        ), row=2, col=i)

layout_num = {**LAYOUT, 'legend': {**LAYOUT['legend'], 'title': {'text': 'Churn Status'}}}
fig.update_layout(**layout_num, title_text='Numeric Features vs Churn', barmode='overlay', height=620)

for r in [1, 2]:
    for c in [1, 2, 3]:
        fig.update_xaxes(gridcolor='#E5E7EB', row=r, col=c)
        fig.update_yaxes(gridcolor='#E5E7EB', row=r, col=c)

fig.show()

print('Mean values by churn status:')
print(f'{"Feature":<20} {"Retained":>12} {"Churned":>12} {"Difference":>12}')
print('-' * 56)
for col in num_cols:
    m_no  = df[df['Churn'] == 'No'][col].mean()
    m_yes = df[df['Churn'] == 'Yes'][col].mean()
    diff  = m_yes - m_no
    arrow = 'up' if diff > 0 else 'down'
    print(f'{col:<20} {m_no:>12.1f} {m_yes:>12.1f}  ({arrow} {abs(diff):.1f})')

Mean values by churn status:
Feature                  Retained      Churned   Difference
--------------------------------------------------------
tenure                       37.6         18.0  (down 19.6)
MonthlyCharges               61.3         74.4  (up 13.2)
TotalCharges               2549.9       1531.8  (down 1018.1)


**Key findings:**
- **Tenure:** Churned customers average around 18 months vs 37 for retained. The first 12 months are the highest-risk window.
- **Monthly Charges:** Churned customers tend to be on more expensive plans.
- **Total Charges:** Lower for churned customers because they left early and didn't accumulate much spend.

### 4.3 Categorical Features vs Churn

Checking which customer segments have the highest churn rates. Red bars = churn rate above 30% (high risk), orange = 15–30%, green = below 15%.

In [18]:
cat_cols = [
    ('Contract',        'Contract Type'),
    ('InternetService', 'Internet Service'),
    ('PaymentMethod',   'Payment Method'),
    ('TechSupport',     'Tech Support'),
    ('OnlineSecurity',  'Online Security'),
    ('SeniorCitizen',   'Senior Citizen'),
]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[title for _, title in cat_cols],
    vertical_spacing=0.18,
    horizontal_spacing=0.09
)

for idx, (col, title) in enumerate(cat_cols):
    r = idx // 3 + 1
    c = idx %  3 + 1

    churn_rate = (
        df.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').mean() * 100)
        .sort_values(ascending=False)
        .reset_index()
    )
    churn_rate.columns = [col, 'ChurnRate']
    bar_colors = churn_rate['ChurnRate'].apply(
        lambda v: RED if v > 30 else ORANGE if v > 15 else GREEN
    )

    fig.add_trace(go.Bar(
        x=churn_rate[col].astype(str),
        y=churn_rate['ChurnRate'],
        marker_color=bar_colors,
        text=[f'{v:.1f}%' for v in churn_rate['ChurnRate']],
        textposition='outside',
        textfont=dict(size=10),
        showlegend=False,
        hovertemplate=f'<b>{title}</b><br>%{{x}}<br>Churn Rate: %{{y:.1f}}%<extra></extra>',
    ), row=r, col=c)

    fig.update_yaxes(title_text='Churn Rate (%)', range=[0, churn_rate['ChurnRate'].max() * 1.22],
                     gridcolor='#E5E7EB', row=r, col=c)
    fig.update_xaxes(tickangle=-20, gridcolor='#E5E7EB', row=r, col=c)

fig.update_layout(**LAYOUT, title_text='Churn Rate by Key Categorical Features', height=660)
fig.show()

print('Churn rate summary by category:')
for col, title in cat_cols:
    rates = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
    print(f'\n{title}:')
    for k, v in rates.sort_values(ascending=False).items():
        flag = '  <- HIGH RISK' if v > 35 else ''
        print(f'  {str(k):<28} {v:5.1f}%{flag}')

Churn rate summary by category:

Contract Type:
  Month-to-month                42.7%  <- HIGH RISK
  One year                      11.3%
  Two year                       2.8%

Internet Service:
  Fiber optic                   41.9%  <- HIGH RISK
  DSL                           19.0%
  No                             7.4%

Payment Method:
  Electronic check              45.3%  <- HIGH RISK
  Mailed check                  19.1%
  Bank transfer (automatic)     16.7%
  Credit card (automatic)       15.2%

Tech Support:
  No                            41.6%  <- HIGH RISK
  Yes                           15.2%
  No internet service            7.4%

Online Security:
  No                            41.8%  <- HIGH RISK
  Yes                           14.6%
  No internet service            7.4%

Senior Citizen:
  1                             41.7%  <- HIGH RISK
  0                             23.6%


**Key findings:**
- Month-to-month contracts churn at **42.7%** vs just 2.8% for two-year contracts — the biggest single signal
- Fiber optic customers churn at **41.9%** — likely a service quality or pricing issue
- Electronic check payment method has **45.3%** churn — manual payers are far less sticky
- Customers without Tech Support or Online Security churn significantly more

### 4.4 Correlation Heatmap

Checking how numeric features relate to each other and to churn. Important for spotting multicollinearity — where two features carry the same information.

In [19]:
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)
numeric_df = df_corr[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']]
corr = numeric_df.corr().round(3)

labels = ['Senior Citizen', 'Tenure', 'Monthly Charges', 'Total Charges', 'Churn']

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels,
    y=labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr.values.round(3),
    texttemplate='<b>%{text}</b>',
    textfont=dict(size=11),
    colorbar=dict(title='Correlation'),
    hovertemplate='<b>%{x} x %{y}</b><br>Correlation: %{z:.3f}<extra></extra>',
))

fig.update_layout(**LAYOUT, title_text='Correlation Matrix — Numeric Features', height=420)
fig.show()

print('Correlation with Churn:')
print(f'  Tenure          : {corr.loc["tenure", "Churn_num"]:+.3f}  (longer tenure = less churn)')
print(f'  Monthly Charges : {corr.loc["MonthlyCharges", "Churn_num"]:+.3f}  (higher charges = more churn)')
print(f'  Total Charges   : {corr.loc["TotalCharges", "Churn_num"]:+.3f}  (lower total = left early)')
print(f'  Senior Citizen  : {corr.loc["SeniorCitizen", "Churn_num"]:+.3f}  (seniors churn slightly more)')
print()
print(f'Multicollinearity warning: tenure and TotalCharges correlate at {corr.loc["tenure", "TotalCharges"]:.3f}')
print('These two features carry overlapping information. Tree models can handle this — linear models cannot.')

Correlation with Churn:
  Tenure          : -0.352  (longer tenure = less churn)
  Monthly Charges : +0.193  (higher charges = more churn)
  Total Charges   : -0.198  (lower total = left early)
  Senior Citizen  : +0.151  (seniors churn slightly more)

Multicollinearity warning: tenure and TotalCharges correlate at 0.826
These two features carry overlapping information. Tree models can handle this — linear models cannot.


### 4.5 Tenure Cohort Analysis

Grouping customers into 12-month tenure buckets to see exactly when in the customer lifecycle churn is highest.

In [22]:
df['TenureBucket'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 36, 48, 60, 72],
    labels=['0-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m']
)

cohort = df.groupby('TenureBucket', observed=True).agg(
    Customers=('customerID', 'count'),
    ChurnRate=('Churn', lambda x: (x == 'Yes').mean() * 100)
).reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=cohort['TenureBucket'].astype(str),
    y=cohort['Customers'],
    name='Customer Count',
    marker_color=BLUE,
    opacity=0.6,
    hovertemplate='<b>Tenure: %{x}</b><br>Customers: %{y:,}<extra></extra>',
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=cohort['TenureBucket'].astype(str),
    y=cohort['ChurnRate'],
    name='Churn Rate %',
    mode='lines+markers+text',
    line=dict(color=RED, width=2.5),
    marker=dict(size=9, color=RED, line=dict(color='white', width=2)),
    text=[f'{v:.1f}%' for v in cohort['ChurnRate']],
    textposition='top center',
    textfont=dict(color=RED, size=11),
    hovertemplate='<b>Tenure: %{x}</b><br>Churn Rate: %{y:.1f}%<extra></extra>',
), secondary_y=True)

layout_cohort = {**LAYOUT}
layout_cohort['legend'] = {**LAYOUT['legend'], 'x': 0.75, 'y': 0.98}

fig.update_layout(
    **layout_cohort,
    title_text='Customer Count and Churn Rate by Tenure Cohort',
    height=430,
)
fig.update_yaxes(title_text='Number of Customers', secondary_y=False, gridcolor='#E5E7EB', color=BLUE)
fig.update_yaxes(title_text='Churn Rate (%)', secondary_y=True, color=RED)
fig.update_xaxes(title_text='Tenure Bucket', gridcolor='#E5E7EB')
fig.show()

print('Cohort breakdown:')
print(cohort.to_string(index=False))

Cohort breakdown:
TenureBucket  Customers  ChurnRate
       0-12m       2175  47.678161
      13-24m       1024  28.710938
      25-36m        832  21.634615
      37-48m        762  19.028871
      49-60m        832  14.423077
      61-72m       1407   6.609808


### 4.6 Churn Pattern Scatter Plot

Visualising individual customers by their tenure and monthly charges, coloured by churn status. This helps identify the highest-risk zone.

In [23]:
fig = px.scatter(
    df,
    x='tenure',
    y='MonthlyCharges',
    color='Churn',
    color_discrete_map={'No': GREEN, 'Yes': RED},
    opacity=0.5,
    hover_data=['customerID', 'Contract', 'InternetService'],
    title='Churn Pattern: Tenure vs Monthly Charges',
    labels={'tenure': 'Tenure (months)', 'MonthlyCharges': 'Monthly Charges ($)'},
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(**LAYOUT, height=460)
fig.update_xaxes(gridcolor='#E5E7EB')
fig.update_yaxes(gridcolor='#E5E7EB')
fig.show()

print('Insight: The top-left cluster (high charges + short tenure) is the highest-risk zone.')
print('New customers on expensive plans are most likely to leave.')

Insight: The top-left cluster (high charges + short tenure) is the highest-risk zone.
New customers on expensive plans are most likely to leave.


---
## 5. Feature Engineering

Before training any model, we need to:
1. Create a few new columns from existing data that might give the model better signals
2. Convert all text columns into numbers (models can only read numbers)
3. Scale the numeric columns so they're on the same scale
4. Split the data into training and test sets
5. Handle the class imbalance using SMOTE

In [24]:
df_model = df.copy()

# Drop columns we don't need for modelling
df_model.drop(['customerID', 'TenureBucket'], axis=1, inplace=True)

# New features built from existing columns
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)
# Normalises total spend by how long the customer has been around

df_model['IsNewCustomer'] = (df_model['tenure'] <= 12).astype(int)
# Flags the highest-risk window we spotted in the cohort analysis

df_model['ChargesPerService'] = df_model['MonthlyCharges'] / (
    df_model[['PhoneService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']]
    .apply(lambda col: col.map({'Yes': 1, 'No': 0,
                                'No internet service': 0,
                                'No phone service': 0}))
    .sum(axis=1) + 1
)
# Monthly charge divided by number of active services — measures value for money

print('New features created: AvgMonthlySpend, IsNewCustomer, ChargesPerService')

New features created: AvgMonthlySpend, IsNewCustomer, ChargesPerService


In [25]:
# Encode binary Yes/No columns as 0 and 1
binary_cols = [
    'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in binary_cols:
    df_model[col] = df_model[col].map({
        'Yes': 1, 'No': 0,
        'No internet service': 0,
        'No phone service': 0
    })

# Encode gender
df_model['gender'] = (df_model['gender'] == 'Male').astype(int)

# One-hot encode columns with more than 2 categories
df_model = pd.get_dummies(
    df_model,
    columns=['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod'],
    drop_first=True
)

# Encode the target column
df_model['Churn'] = (df_model['Churn'] == 'Yes').astype(int)

print(f'Dataset shape after encoding: {df_model.shape}')
print(f'Total features for model    : {df_model.shape[1] - 1}')

Dataset shape after encoding: (7043, 28)
Total features for model    : 27


In [26]:
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Stratified split — preserves the 73.5/26.5 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features — required for Logistic Regression
# Fit on training data only, then apply the same scaling to test
scaler = StandardScaler()
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'ChargesPerService']
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features]  = scaler.transform(X_test[num_features])

# Apply SMOTE to training set only — creates synthetic minority-class samples
# Never apply SMOTE before splitting — that would contaminate the test set
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Train set : {X_train.shape[0]:,} rows | Churn rate: {y_train.mean()*100:.1f}%')
print(f'Test set  : {X_test.shape[0]:,} rows  | Churn rate: {y_test.mean()*100:.1f}%')
print()
print(f'Before SMOTE: {dict(y_train.value_counts())}')
print(f'After SMOTE : {dict(pd.Series(y_train_sm).value_counts())}')

Train set : 5,634 rows | Churn rate: 26.5%
Test set  : 1,409 rows  | Churn rate: 26.5%

Before SMOTE: {0: np.int64(4139), 1: np.int64(1495)}
After SMOTE : {0: np.int64(4139), 1: np.int64(4139)}


---
## 6. Modelling

We train three different models and compare them. The reason for training multiple is simple — no algorithm is always best, and comparing them gives us a fair picture of what works for this specific dataset.

- **Logistic Regression** — simple baseline, highly interpretable
- **Random Forest** — handles non-linear patterns well, gives feature importance
- **XGBoost** — generally the best performer on tabular data, industry standard

In [27]:
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42, C=0.1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        scale_pos_weight=3, random_state=42,
        eval_metric='logloss', verbosity=0
    ),
}

results    = {}
cv_results = {}
cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_sm, y_train_sm)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    rep = classification_report(y_test, y_pred, output_dict=True)

    cv_f1 = cross_val_score(model, X_train_sm, y_train_sm, cv=cv, scoring='f1', n_jobs=-1)

    results[name]    = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
                        'accuracy': acc, 'f1': f1, 'auc': auc, 'report': rep}
    cv_results[name] = cv_f1

    print(f'  Accuracy : {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')
    print(f'  CV F1    : {cv_f1.mean():.4f} +/- {cv_f1.std():.4f}')
    print()

Training Logistic Regression...
  Accuracy : 0.7445 | F1: 0.6053 | AUC: 0.8301
  CV F1    : 0.7925 +/- 0.0060

Training Random Forest...
  Accuracy : 0.7544 | F1: 0.6198 | AUC: 0.8365
  CV F1    : 0.8161 +/- 0.0066

Training XGBoost...
  Accuracy : 0.6920 | F1: 0.5967 | AUC: 0.8232
  CV F1    : 0.8170 +/- 0.0040



### 6.1 Model Comparison

Comparing all three models side by side on the three metrics that matter most for this problem.

In [28]:
model_names = list(results.keys())
short_names = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
bar_colors  = [BLUE, RED, GREEN]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Accuracy', 'F1-Score (Churn)', 'ROC-AUC'],
    horizontal_spacing=0.10
)

for col_idx, metric_key in enumerate(['accuracy', 'f1', 'auc'], 1):
    vals = [results[m][metric_key] for m in model_names]
    fig.add_trace(go.Bar(
        x=short_names,
        y=vals,
        marker_color=bar_colors,
        text=[f'{v:.4f}' for v in vals],
        textposition='outside',
        textfont=dict(size=11),
        showlegend=False,
        width=0.4,
        hovertemplate='<b>%{x}</b><br>Score: %{y:.4f}<extra></extra>',
    ), row=1, col=col_idx)
    fig.update_yaxes(range=[min(vals) * 0.92, min(max(vals) * 1.08, 1.0)],
                     gridcolor='#E5E7EB', row=1, col=col_idx)
    fig.update_xaxes(gridcolor='#E5E7EB', row=1, col=col_idx)

fig.update_layout(**LAYOUT, title_text='Model Performance Comparison', height=400)
fig.show()

# Summary table
comparison = pd.DataFrame({
    name: {
        'Accuracy':           round(res['accuracy'], 4),
        'F1-Score':           round(res['f1'], 4),
        'ROC-AUC':            round(res['auc'], 4),
        'CV F1 (mean)':       round(cv_results[name].mean(), 4),
        'Precision (Churn)':  round(res['report']['1']['precision'], 4),
        'Recall (Churn)':     round(res['report']['1']['recall'], 4),
    }
    for name, res in results.items()
}).T

print(comparison.to_string())

best_name  = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]
print(f'\nBest model: {best_name} (ROC-AUC = {best_model["auc"]:.4f})')

                     Accuracy  F1-Score  ROC-AUC  CV F1 (mean)  Precision (Churn)  Recall (Churn)
Logistic Regression    0.7445    0.6053   0.8301        0.7925             0.5130          0.7380
Random Forest          0.7544    0.6198   0.8365        0.8161             0.5261          0.7540
XGBoost                0.6920    0.5967   0.8232        0.8170             0.4573          0.8583

Best model: Random Forest (ROC-AUC = 0.8365)


---
## 7. Evaluation

Now we take a close look at how the best model actually performs — not just as a single number, but through multiple lenses: the confusion matrix, ROC curve, precision-recall tradeoff, cross-validation stability, and which features it relied on most.

### 7.1 Confusion Matrix

In [29]:
cm = confusion_matrix(y_test, best_model['y_pred'])
tn, fp, fn, tp = cm.ravel()

fig = go.Figure(go.Heatmap(
    z=[[tn, fp], [fn, tp]],
    x=['Predicted: No Churn', 'Predicted: Churn'],
    y=['Actual: No Churn', 'Actual: Churn'],
    colorscale=[[0, '#EBF5FB'], [1, BLUE]],
    showscale=False,
    text=[[f'<b>{tn:,}</b>', f'<b>{fp:,}</b>'], [f'<b>{fn:,}</b>', f'<b>{tp:,}</b>']],
    texttemplate='%{text}',
    textfont=dict(size=18),
    hovertemplate='<b>%{y} x %{x}</b><br>Count: %{z:,}<extra></extra>',
))

fig.update_layout(
    **LAYOUT,
    title_text=f'Confusion Matrix — {best_name}',
    height=360,
    xaxis=dict(side='bottom'),
)
fig.show()

print(f'True Positives  (churners caught)      : {tp:,}')
print(f'False Negatives (churners missed)      : {fn:,}  <- most costly mistake')
print(f'True Negatives  (loyal customers OK)   : {tn:,}')
print(f'False Positives (unnecessary alerts)   : {fp:,}')
print()
print(f'Recall: {tp/(tp+fn):.1%} of actual churners were identified by the model.')

True Positives  (churners caught)      : 282
False Negatives (churners missed)      : 92  <- most costly mistake
True Negatives  (loyal customers OK)   : 781
False Positives (unnecessary alerts)   : 254

Recall: 75.4% of actual churners were identified by the model.


### 7.2 ROC Curves — All Models

In [30]:
fig = go.Figure()
colors_roc = [BLUE, RED, GREEN]

for name, color in zip(model_names, colors_roc):
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    auc_val      = results[name]['auc']
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{name} (AUC = {auc_val:.4f})',
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{name}</b><br>FPR: %{{x:.3f}}<br>TPR: %{{y:.3f}}<extra></extra>',
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random baseline (AUC = 0.5)',
    line=dict(color=GREY, width=1.5, dash='dash'),
))

layout_roc = {**LAYOUT, 'legend': {**LAYOUT['legend'], 'x': 0.55, 'y': 0.12}}
fig.update_layout(
    **layout_roc,
    title_text='ROC Curves — All Models',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=460,
    xaxis=dict(range=[0, 1], gridcolor='#E5E7EB'),
    yaxis=dict(range=[0, 1.02], gridcolor='#E5E7EB'),
)
fig.show()

### 7.3 Precision-Recall Curve

In [31]:
fig = go.Figure()

for name, color in zip(model_names, colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, results[name]['y_prob'])
    fig.add_trace(go.Scatter(
        x=rec, y=prec,
        mode='lines',
        name=name,
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{name}</b><br>Recall: %{{x:.3f}}<br>Precision: %{{y:.3f}}<extra></extra>',
    ))

baseline = y_test.mean()
fig.add_hline(
    y=baseline,
    line_dash='dash',
    line_color=GREY,
    annotation_text=f'Baseline ({baseline:.2f})',
    annotation_position='bottom right'
)

fig.update_layout(
    **LAYOUT,
    title_text='Precision-Recall Curve — All Models',
    xaxis_title='Recall',
    yaxis_title='Precision',
    height=420,
    xaxis=dict(range=[0, 1], gridcolor='#E5E7EB'),
    yaxis=dict(range=[0, 1.05], gridcolor='#E5E7EB'),
)
fig.show()

### 7.4 Cross-Validation F1 Scores

In [32]:
fig = go.Figure()
fold_labels = [f'Fold {i+1}' for i in range(5)]

for name, color in zip(model_names, colors_roc):
    cv_data  = cv_results[name]
    mean_val = cv_data.mean()

    fig.add_trace(go.Bar(
        x=fold_labels,
        y=cv_data,
        name=name,
        marker_color=color,
        opacity=0.8,
        text=[f'{v:.3f}' for v in cv_data],
        textposition='outside',
        hovertemplate=f'<b>{name}</b><br>%{{x}}<br>F1: %{{y:.4f}}<extra></extra>',
    ))
    fig.add_hline(
        y=mean_val,
        line_dash='dot',
        line_color=color,
        opacity=0.7,
        annotation_text=f'{name} mean = {mean_val:.3f}',
        annotation_position='top left' if name != 'XGBoost' else 'top right'
    )

fig.update_layout(
    **LAYOUT,
    title_text='5-Fold Cross-Validation F1 Scores',
    xaxis_title='Fold',
    yaxis_title='F1 Score',
    barmode='group',
    height=430,
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

### 7.5 Feature Importance

Which features did the model rely on most when deciding whether a customer would churn? This is one of the most valuable outputs — it tells us what to actually fix.

In [33]:
if hasattr(best_model['model'], 'feature_importances_'):
    imp = pd.Series(
        best_model['model'].feature_importances_,
        index=X_train.columns
    ).nlargest(20).sort_values()
    imp_label = 'Importance Score'
else:
    imp = pd.Series(
        np.abs(best_model['model'].coef_[0]),
        index=X_train.columns
    ).nlargest(20).sort_values()
    imp_label = 'Absolute Coefficient'

bar_colors = [RED if i == len(imp) - 1 else BLUE for i in range(len(imp))]

fig = go.Figure(go.Bar(
    x=imp.values,
    y=imp.index,
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color='white', width=0.5)),
    text=[f'{v:.4f}' for v in imp.values],
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>' + imp_label + ': %{x:.4f}<extra></extra>',
))

layout_fi = {k: v for k, v in LAYOUT.items() if k != 'margin'}
fig.update_layout(
    **layout_fi,
    title_text=f'Top 20 Feature Importances — {best_name}',
    xaxis_title=imp_label,
    yaxis_title='Feature',
    height=560,
    margin=dict(l=200, t=70, r=80, b=50),
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

print('Top 10 features driving churn:')
for feat, val in imp.tail(10).sort_values(ascending=False).items():
    print(f'  {feat:<40} {val:.4f}')

Top 10 features driving churn:
  ChargesPerService                        0.1776
  tenure                                   0.1315
  Contract_Two year                        0.0835
  PaymentMethod_Electronic check           0.0755
  MonthlyCharges                           0.0720
  TotalCharges                             0.0713
  AvgMonthlySpend                          0.0655
  InternetService_Fiber optic              0.0558
  IsNewCustomer                            0.0378
  OnlineSecurity                           0.0319


### 7.6 Full Classification Reports

In [34]:
for name in model_names:
    print('-' * 52)
    print(f'{name}')
    print('-' * 52)
    print(classification_report(
        y_test, results[name]['y_pred'],
        target_names=['No Churn', 'Churned']
    ))
    print()

----------------------------------------------------
Logistic Regression
----------------------------------------------------
              precision    recall  f1-score   support

    No Churn       0.89      0.75      0.81      1035
     Churned       0.51      0.74      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.74      0.71      1409
weighted avg       0.79      0.74      0.76      1409


----------------------------------------------------
Random Forest
----------------------------------------------------
              precision    recall  f1-score   support

    No Churn       0.89      0.75      0.82      1035
     Churned       0.53      0.75      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.75      0.77      1409


----------------------------------------------------
XGBoost
-------------------------------------

---
## 8. Business Insights

Now we zoom out from model metrics and look at the data from a business perspective. These charts are designed to be shareable with non-technical stakeholders — the goal is clear, actionable findings.

### 8.1 Risk Segment Heatmap

Which combination of contract type and internet service is the most dangerous? This tells the retention team exactly where to focus first.

In [35]:
pivot = (
    df.groupby(['Contract', 'InternetService'])['Churn']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .unstack()
    .round(1)
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='YlOrRd',
    text=pivot.values.round(1),
    texttemplate='<b>%{text}%</b>',
    textfont=dict(size=13),
    colorbar=dict(title='Churn Rate %'),
    hovertemplate='<b>%{y} | %{x}</b><br>Churn Rate: %{z:.1f}%<extra></extra>',
))

fig.update_layout(
    **LAYOUT,
    title_text='Churn Rate % — Contract Type x Internet Service',
    xaxis_title='Internet Service',
    yaxis_title='Contract Type',
    height=360,
)
fig.show()

### 8.2 Churn by Payment Method and Contract Type

In [36]:
pay_churn = (
    df.groupby('PaymentMethod')['Churn']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .reset_index()
)
pay_churn.columns = ['PaymentMethod', 'ChurnRate']
pay_churn = pay_churn.sort_values('ChurnRate', ascending=False)

cont_churn = (
    df.groupby('Contract')['Churn']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .reset_index()
)
cont_churn.columns = ['Contract', 'ChurnRate']
cont_churn = cont_churn.sort_values('ChurnRate', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Churn Rate by Payment Method', 'Churn Rate by Contract Type'],
    horizontal_spacing=0.12
)

for df_plot, col_name, col_idx in [
    (pay_churn, 'PaymentMethod', 1),
    (cont_churn, 'Contract', 2)
]:
    bar_colors = [
        RED if v > 30 else ORANGE if v > 15 else GREEN
        for v in df_plot['ChurnRate']
    ]
    fig.add_trace(go.Bar(
        x=df_plot[col_name],
        y=df_plot['ChurnRate'],
        marker_color=bar_colors,
        text=[f'{v:.1f}%' for v in df_plot['ChurnRate']],
        textposition='outside',
        showlegend=False,
        width=0.5,
        hovertemplate='<b>%{x}</b><br>Churn Rate: %{y:.1f}%<extra></extra>',
    ), row=1, col=col_idx)
    fig.update_yaxes(
        title_text='Churn Rate (%)',
        range=[0, df_plot['ChurnRate'].max() * 1.2],
        gridcolor='#E5E7EB', row=1, col=col_idx
    )
    fig.update_xaxes(tickangle=-20, gridcolor='#E5E7EB', row=1, col=col_idx)

fig.update_layout(**LAYOUT, title_text='Churn by Payment Method and Contract', height=420)
fig.show()

### 8.3 Predicted Churn Probability Distribution

How confident is the model in its predictions? A well-performing model should clearly separate churners (pushing them toward probability 1.0) from retained customers (pushing them toward 0.0).

In [37]:
fig = go.Figure()

for label, color, name_str in [(0, GREEN, 'No Churn'), (1, RED, 'Churned')]:
    mask = y_test == label
    fig.add_trace(go.Histogram(
        x=best_model['y_prob'][mask],
        name=name_str,
        marker_color=color,
        opacity=0.65,
        nbinsx=40,
        hovertemplate=f'<b>{name_str}</b><br>Probability: %{{x:.2f}}<br>Count: %{{y}}<extra></extra>',
    ))

fig.add_vline(
    x=0.5,
    line_dash='dash',
    line_color=GREY,
    annotation_text='Decision threshold (0.5)',
    annotation_position='top right'
)

layout_prob = {**LAYOUT, 'legend': {**LAYOUT['legend'], 'title': {'text': 'Actual Status'}}}
fig.update_layout(
    **layout_prob,
    title_text=f'Predicted Churn Probability Distribution — {best_name}',
    xaxis_title='Predicted Churn Probability',
    yaxis_title='Count',
    barmode='overlay',
    height=400,
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

print('The more the two distributions separate, the better the model.')
print('Churned (red) should pile up near 1.0, retained (green) near 0.0.')

The more the two distributions separate, the better the model.
Churned (red) should pile up near 1.0, retained (green) near 0.0.


---
## 9. Export Predictions

The final step is exporting the model's predictions as a CSV.

In [41]:
test_idx  = X_test.index
export_df = df.loc[test_idx, [
    'customerID', 'gender', 'SeniorCitizen', 'tenure',
    'Contract', 'InternetService', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges', 'Churn'
]].copy()

export_df['Churn_Actual']      = export_df['Churn'].map({'Yes': 1, 'No': 0})
export_df['Churn_Predicted']   = best_model['y_pred']
export_df['Churn_Probability'] = best_model['y_prob'].round(4)
export_df['Risk_Level']        = pd.cut(
    best_model['y_prob'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)
export_df.drop('Churn', axis=1, inplace=True)

export_df.to_csv('churn_predictions.csv', index=False)

print('Saved: churn_predictions.csv')
print(f'Total customers  : {len(export_df):,}')
print(f'Predicted churners: {export_df["Churn_Predicted"].sum():,}')
print()
print('Risk Level breakdown:')
print(export_df['Risk_Level'].value_counts().to_string())
print()
print('Preview:')
export_df.head()

Saved: churn_predictions.csv
Total customers  : 1,409
Predicted churners: 536

Risk Level breakdown:
Risk_Level
Low Risk       659
High Risk      404
Medium Risk    346

Preview:


,customerID,gender,SeniorCitizen,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,TotalCharges,Churn_Actual,Churn_Predicted,Churn_Probability,Risk_Level
437,4376-KFVRS,Male,0,72,Two year,Fiber optic,Credit card (automatic),114.05,8468.20,0,0,0.0205,Low Risk
2280,2754-SDJRD,Female,1,8,Month-to-month,Fiber optic,Credit card (automatic),100.15,908.55,0,1,0.8876,High Risk
2235,9917-KWRBE,Female,0,41,One year,DSL,Credit card (automatic),78.35,3211.20,0,0,0.1515,Low Risk
4460,0365-GXEZS,Male,0,18,Month-to-month,Fiber optic,Electronic check,78.20,1468.75,0,1,0.5213,Medium Risk
3761,9385-NXKDA,Female,0,72,Two year,DSL,Credit card (automatic),82.65,5919.35,0,0,0.0469,Low Risk


---
## 10. Summary

A full recap of everything this notebook did and what it found.

In [40]:
print('=' * 60)
print('  TELCO CHURN ANALYSIS — FINAL SUMMARY')
print('=' * 60)

print(f"""
Dataset
  Source      : IBM Telco Customer Churn
  Rows        : {df.shape[0]:,}
  Features    : {df.shape[1]} original + 3 engineered
  Churn Rate  : 26.5%  (handled with SMOTE)

Data Quality Issues Fixed
  TotalCharges stored as string  ->  converted to float
  11 hidden nulls (tenure = 0)   ->  filled with 0
  customerID dropped             ->  not a predictive feature

Model Results
  Logistic Regression  :  F1 = {results['Logistic Regression']['f1']:.4f}  |  AUC = {results['Logistic Regression']['auc']:.4f}
  Random Forest        :  F1 = {results['Random Forest']['f1']:.4f}  |  AUC = {results['Random Forest']['auc']:.4f}
  XGBoost              :  F1 = {results['XGBoost']['f1']:.4f}  |  AUC = {results['XGBoost']['auc']:.4f}

Best Model  :  {best_name}
  F1-Score   :  {best_model['f1']:.4f}
  ROC-AUC    :  {best_model['auc']:.4f}
  Recall     :  {best_model['report']['1']['recall']:.4f}  (caught {best_model['report']['1']['recall']*100:.1f}% of actual churners)
  Precision  :  {best_model['report']['1']['precision']:.4f}

Top Churn Drivers
  1. Contract type   —  month-to-month = 42.7% churn vs 2.8% on 2-year plans
  2. Tenure          —  first 12 months are the highest risk window
  3. Internet service —  fiber optic customers churn at 41.9%
  4. Payment method  —  electronic check = 45.3% churn rate
  5. Support services —  customers without TechSupport churn significantly more

Business Recommendations
  - Offer incentives to convert month-to-month customers to annual plans
  - Run targeted retention campaigns for customers in their first 12 months
  - Investigate fiber optic service quality and pricing
  - Encourage auto-payment methods over electronic check
  - Bundle TechSupport and OnlineSecurity into base plans

""")

print('=' * 60)

  TELCO CHURN ANALYSIS — FINAL SUMMARY

Dataset
  Source      : IBM Telco Customer Churn
  Rows        : 7,043
  Features    : 22 original + 3 engineered
  Churn Rate  : 26.5%  (handled with SMOTE)

Data Quality Issues Fixed
  TotalCharges stored as string  ->  converted to float
  11 hidden nulls (tenure = 0)   ->  filled with 0
  customerID dropped             ->  not a predictive feature

Model Results
  Logistic Regression  :  F1 = 0.6053  |  AUC = 0.8301
  Random Forest        :  F1 = 0.6198  |  AUC = 0.8365
  XGBoost              :  F1 = 0.5967  |  AUC = 0.8232

Best Model  :  Random Forest
  F1-Score   :  0.6198
  ROC-AUC    :  0.8365
  Recall     :  0.7540  (caught 75.4% of actual churners)
  Precision  :  0.5261

Top Churn Drivers
  1. Contract type   —  month-to-month = 42.7% churn vs 2.8% on 2-year plans
  2. Tenure          —  first 12 months are the highest risk window
  3. Internet service —  fiber optic customers churn at 41.9%
  4. Payment method  —  electronic check = 